In [0]:
%sh
rm -rf /dbfs/device_stream
mkdir -p /dbfs/device_stream
curl -L -o /dbfs/device_stream/device_data.csv https://github.com/MicrosoftLearning/mslearn-databricks/raw/main/data/device_data.csv

In [0]:
 from pyspark.sql.functions import *
 from pyspark.sql.types import *

 # Define the schema for the incoming data
 schema = StructType([
     StructField("device_id", StringType(), True),
     StructField("timestamp", TimestampType(), True),
     StructField("temperature", DoubleType(), True),
     StructField("humidity", DoubleType(), True)
 ])

 # Read streaming data from folder
 inputPath = '/device_stream/'
 iotstream = spark.readStream.schema(schema).option("header", "true").csv(inputPath)
 print("Source stream created...")

 # Write the data to a Delta table
 query = (iotstream
          .writeStream
          .format("delta")
          .option("checkpointLocation", "/tmp/checkpoints/iot_data")
          .start("/tmp/delta/iot_data"))